# 伟大建筑讲解 + 公司宣传册生成

## 练习目标（理念）

用 **Gradio** 做两个可交互的小应用，把第 1 周学到的「调模型 + 网页抓取 + 流式输出」串起来：

1. **伟大建筑讲解**：在界面里输入问题，用本地 **Ollama / Llama3.2** 流式讲解建筑（面向普通人 vs 土木方向可换示例）
2. **公司宣传册生成**：输入公司名 + 落地页 URL，先用 `BeautifulSoup` 抓取网页正文，再选用 **GPT** 或 **Ollama** 流式生成宣传册

这是典型的「工具链」练习：抓取 → 拼 prompt → Chat Completions（可流式）→ Gradio 边生成边显示。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` / Ollama 兼容的同一套 SDK |
| `messages`（system / user） | `system_message` 定角色，user 放建筑问题或宣传册需求 |
| 流式输出 `stream=True` | `stream_gpt` / `stream_ollama` 里逐步 `yield` 累积文本 |
| 网页抓取 | `requests` + `BeautifulSoup`：`fetch_website_contents` |
| Gradio UI | `gr.Interface` 把函数挂成网页表单 |

## 怎么跑

1. 从上到下依次运行单元格（Shift+Enter）
2. 准备 `.env`：至少有 `OPENAI_API_KEY`（若只用本地 Ollama 可先跑建筑讲解）
3. 本机已安装并启动 Ollama，执行过 `ollama pull llama3.2`
4. 先跑「伟大建筑」界面；再跑「宣传册」界面，对比 GPT 与 OLLAMA



In [ ]:
# ========== 导入：后面抓取、调模型、展示都要用到这些工具箱 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如各家 API Key
import os
# 导入标准库 json：本练习后续若解析结构化响应可用（先导入备用）
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：既可连云端 GPT，也可 base_url 指向本地 Ollama 兼容接口
from openai import OpenAI
# 从 IPython.display 导入展示工具：在笔记本里用 Markdown 漂亮显示文本
from IPython.display import Markdown, display



In [ ]:
# ========== 网页抓取：用 requests + BeautifulSoup 取标题/正文/链接 ==========

# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可查询的文档树
from bs4 import BeautifulSoup
# 导入标准库 requests：发 HTTP GET，下载网页
import requests


# 抓取网页用的标准请求头（User-Agent）：模拟浏览器，降低被站点直接拒绝的概率
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """返回给定 URL 网页的标题与正文；截断到 2,000 字符作为合理上限（控制 prompt 长度）。"""
    # GET 下载页面；带上 headers，避免部分站点对「脚本默认 UA」直接 403
    response = requests.get(url, headers=headers)
    # 用 html.parser 把字节内容解析成 soup 对象
    soup = BeautifulSoup(response.content, "html.parser")
    # 有 <title> 就取字符串，否则给占位文案（英文占位保留：可运行字符串）
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        # 删掉 script / style / img / input 等对「宣传册文字」无用的节点
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        # 取出纯文本：用换行分隔，strip 去掉首尾空白
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        # 没有 body 时正文为空串
        text = ""
    # 标题 + 空行 + 正文，再切片前 2000 字符，防止塞进模型的 prompt 过长
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """返回网页上的超链接列表。会再解析一次（实验代码求简单，可自行改成类复用 soup）。"""
    # 再次 GET + 解析（与上面函数重复抓取——课程里为清晰刻意拆开）
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    # 找出所有 <a>，取出 href 属性（可能是相对路径或 None）
    links = [link.get("href") for link in soup.find_all("a")]
    # 过滤掉空/None，只保留真有地址的链接
    return [link for link in links if link]



## 网页抓取辅助函数（上一格代码在做什么）

上一格定义了两个函数，后面「宣传册生成」会用到：

- **`fetch_website_contents(url)`**  
  用 `requests.get` 下载页面 → `BeautifulSoup` 解析 → 去掉 `script` / `style` 等噪音 → 取出标题 + 正文，**截断到 2000 字符**，方便塞进 prompt。

- **`fetch_website_links(url)`**  
  同样抓取并解析，收集所有 `<a href=...>`。课程里故意再解析一次（简单优先）；你也可以改成类，一次解析同时拿正文和链接。

请求头里的 **User-Agent** 很重要：很多站点会拦「不像浏览器」的默认客户端。



In [ ]:
# ========== 导入 Gradio 与 ollama 包（界面 + 本地模型生态） ==========

# 导入 gradio 并起别名 gr：几行代码就能把 Python 函数变成网页 UI
import gradio as gr # oh yeah!
# 导入 ollama 包：本练习后面主要用 OpenAI 兼容客户端调 Ollama；此导入保留原逻辑
import ollama



In [ ]:
# ========== 拉取本地模型：确保本机有 llama3.2（需已安装并启动 Ollama） ==========
# Jupyter 魔法命令 !：在 shell 里执行 ollama pull；模型名字符串必须与本地一致
!ollama pull llama3.2



In [ ]:
# ========== 用 OpenAI 兼容协议连接本地 Ollama（不是云端 OpenAI） ==========

# Ollama 提供的 OpenAI 风格 Base URL：/v1 路径下可用 chat.completions
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 创建指向本地的 OpenAI 客户端；api_key 对 Ollama 通常任意非空即可（这里用 'ollama'）
# 注意：变量名也叫 ollama，会覆盖上一格 import 的 ollama 包名——后面 stream_ollama 用的是这个客户端
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')



In [ ]:
# ========== 加载 .env 并检查各家 API Key 是否存在（不打印完整密钥） ==========

# override=True：.env 里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenAI / Anthropic / Google 的 Key（后两者本练习未必用到，但统一检查）
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# 有 Key 就打印前缀做「存在性」确认；没有就提示未设置（文案保留英文，与原输出风格一致）
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")



In [ ]:
# （空单元格占位：原笔记本此处无代码；可在此继续实验，勿删以免打乱运行顺序习惯）



In [ ]:
# ========== 创建默认 OpenAI 云端客户端（读环境变量 OPENAI_API_KEY） ==========

# 不传 api_key 时，SDK 会自动从环境变量 OPENAI_API_KEY 读取
openai = OpenAI()




In [ ]:
# ========== 非流式 GPT 调用（完整返回后再给调用方） ==========

# system 角色的固定指令：保留英文——这是发给模型的 prompt，改译会改变回答风格
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    # 组装 messages：system 定人设，user 放用户输入
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    # 一次性拿到完整 completion（默认 stream=False）
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    # 取出第一条 choice 里助手回复的文本内容
    return response.choices[0].message.content



In [ ]:
# ========== 再次定义 message_gpt（与上一格等价；保留原笔记本结构） ==========

# system prompt 同样保留英文可运行字符串
system_message = "You are a helpful assistant"

def message_gpt(prompt):
    # 非流式调用 GPT：等整段生成完再返回
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    # model id 保留原样：gpt-4.1-mini
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    # 返回助手消息正文
    return response.choices[0].message.content




In [ ]:
# ========== 流式 GPT：边生成边 yield「到目前为止的完整文本」 ==========

def stream_gpt(prompt):
    # 流式调用 GPT，逐步 yield 累积文本（Gradio 适合这种「越来越长」的更新方式）
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    # stream=True：服务端持续推送增量 delta，不必等整段结束
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    # result：把每一块增量拼起来；每次 yield 当前全文，方便界面刷新
    result = ""
    for chunk in stream:
        # delta.content 可能是 None（某些控制块），用 or "" 避免把 None 拼进去
        result += chunk.choices[0].delta.content or ""
        yield result




In [ ]:
# ========== 流式 Ollama：接口形状与 OpenAI SDK 相同，只是客户端指向本地 ==========

def stream_ollama(prompt):
    # 流式调用本地 Ollama（变量 ollama 是 OpenAI 兼容客户端，不是 import 的包）
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    # model 名必须与本机 ollama list / pull 的名字一致：llama3.2
    stream = ollama.chat.completions.create(
        model='llama3.2',
        messages=messages,
        stream=True
    )
    # 与 stream_gpt 相同：累积全文并逐步 yield
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result




In [ ]:
# ========== Gradio：伟大建筑讲解界面（输入问题 → 流式调用 stream_ollama） ==========

# 多行文本框：label/info/examples 保留英文原样（界面文案与可运行示例字符串）
message_input = gr.Textbox(label="Your message:", info="Enter a message for llama3.2", lines=7)
# 输出用 Markdown 组件渲染模型回复
message_output = gr.Markdown(label="Response:")

# Interface：把 stream_ollama 挂成 Web UI；flagging_mode="never" 关闭标记反馈按钮
view = gr.Interface(
    fn=stream_ollama,
    title="Great Architecture", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Tanjore Temple architecture to a layperson",
        "Explain the Tanjore  Architecture to an aspiring Civil engineer",
        ], 
    flagging_mode="never"
    )
# launch：在笔记本里启动本地 Gradio 服务并展示界面
view.launch()




In [ ]:
# ========== 宣传册生成：抓取落地页 → 拼 prompt → 按模型分流流式输出 ==========

def stream_brochure(company_name, url, model):
    # 根据公司名与落地页内容流式生成宣传册
    # 先 yield 空串：让 Gradio 立刻清空/刷新输出区
    yield ""
    # 英文 prompt 保留：影响模型行为的可运行字符串不翻译
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    # 把抓到的标题+正文追加进 prompt
    prompt += fetch_website_contents(url)
    # 按下拉框选项分流：GPT 走云端，OLLAMA 走本地
    if model=="GPT":
        result = stream_gpt(prompt)
    elif model=="OLLAMA":
        result = stream_ollama(prompt)
    else:
        # 未知模型名：抛错（文案保留英文）
        raise ValueError("Unknown model")
    # yield from：把子生成器里每一段累积文本继续向外抛
    yield from result




In [ ]:
# ========== Gradio：宣传册生成器（公司名 + URL + 模型选择） ==========

# 三个输入：公司名、落地页 URL、模型下拉（GPT / OLLAMA）；界面文案保留英文原样
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GPT", "OLLAMA"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Bronx Zoo", "https://Bronxzoo.com", "GPT"],
            ["A2B Restaurant", "https://a2bva.com/", "OLLAMA"]
        ], 
    flagging_mode="never"
    )
# 启动第二个 Gradio 应用（宣传册）
view.launch()


